# Lens-pair dataset — random close Gaia/GOST scanning laws

This notebook creates synthetic **positive and negative lens-pair datasets**.

The default intrinsic quasar model is now a **Damped Random Walk (DRW)**. Amoeba
remains optional, and DRW is also the documented fallback if Amoeba is unavailable.

Core rule: for **every pair**, positive and negative, we draw a random sky position
and a second close object. We then query GOST for object A and object B separately.
Positives and negatives therefore have the same sky, separation, and scanning-law
construction.

Outputs:
- `metadata.csv`
- `observations.csv`
- `continuous_curves.csv`
- `config.json`
- for the small run: one plot per pair showing continuous curves, GOST-sampled
  points, and microlensing curves.


## 0. Optional install

In [ ]:
# Uncomment if needed
# import sys
# !{sys.executable} -m pip install -U numpy pandas scipy matplotlib astropy requests amoeba-agn


## 1. Imports and configuration

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

# Standard Python tools used to save configuration files, pause between
# web requests, build cache identifiers, manage warnings, and create paths.
import hashlib
import json
import time
import warnings
from io import BytesIO
from pathlib import Path

# Numerical arrays, tables, figures, and HTTP requests to the GOST service.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

# Scientific tools used for smooth microlensing curves, astronomical dates,
# and the VOTable format returned by GOST.
from astropy.io.votable import parse_single_table
from astropy.time import Time
from scipy.ndimage import gaussian_filter1d

# Hide RuntimeWarning messages produced by harmless intermediate NaN values.
# Set this line to a comment while debugging numerical problems.
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. CHOOSE WHICH DATASET RUNS TO EXECUTE
# ============================================================

# True runs the small six-pair example at the end of the notebook.
# Run this first to check the installation and inspect the figures.
RUN_SMALL = True

# True runs the large 1,000-pair dataset at the end of the notebook.
# Keep this False until the small example works because the large run makes
# many GOST requests and can require considerable computation time.
RUN_BIG = False


# ============================================================
# 2. CHOOSE THE INTRINSIC QUASAR GENERATOR
# ============================================================

# False uses the DRW model directly. This is the recommended default because
# it is fast, reproducible, and includes a physical memory timescale.
# True tries Amoeba first and uses DRW only if Amoeba fails.
USE_AMOEBA_IF_AVAILABLE = False

# True allows the notebook to continue with DRW if Amoeba is unavailable or
# fails. False stops and displays the Amoeba error instead. This setting has
# no effect when USE_AMOEBA_IF_AVAILABLE is False.
ALLOW_DRW_FALLBACK = True


# ============================================================
# 3. OUTPUT DIRECTORIES
# ============================================================

# Main directory containing every generated file. Relative paths are created
# inside the directory from which JupyterLab was started.
OUTPUT_ROOT = Path("lens_pair_dataset_random_close_scanning")

# Cached GOST responses are stored here. The same sky query can then be reused
# without making another web request.
GOST_CACHE_DIR = OUTPUT_ROOT / "gost_cache"

# Separate output directories for the small test and large production run.
SMALL_DIR = OUTPUT_ROOT / "small"
BIG_DIR = OUTPUT_ROOT / "big"

# Create all directories automatically when they do not already exist.
for path in [OUTPUT_ROOT, GOST_CACHE_DIR, SMALL_DIR, BIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. GOST REQUEST SAFETY SETTINGS
# ============================================================

# Pause after a successful online GOST request. Increasing this value reduces
# the request rate but makes a new uncached dataset slower to generate.
GOST_SLEEP_BETWEEN_CALLS_SEC = 0.2

# Number of times one failed sky-position request is attempted again.
MAX_GOST_RETRIES_PER_OBJECT = 2

# Maximum number of random sky positions tried when one complete pair cannot
# be generated, for example because GOST returns too few usable observations.
MAX_SKY_TRIES_PER_PAIR = 30


# ============================================================
# 5. PARAMETERS SHARED BY THE SMALL AND LARGE DATASETS
# ============================================================

BASE_CONFIG = dict(
    # Duration of each continuous synthetic light curve, in days. GOST visits
    # outside this simulated interval are ignored. 2,000 days is about 5.5 years.
    n_days=2000,

    # Time spacing of the continuous synthetic curve. 1.0 means one value per day.
    # A smaller value gives a denser curve but increases memory and computation.
    cadence_days=1.0,

    # Extra history generated before day zero so a delayed curve remains defined.
    # This should normally be at least as large as delay_max_days. The legacy
    # value 400 is retained here, although delays up to 700 days are requested.
    max_delay_days=400,

    # Beginning and end of the interval requested from the Gaia GOST service.
    gost_start_utc="2014-07-25T00:00:00",
    gost_end_utc="2025-01-15T00:00:00",

    # Maximum number of records that one GOST query is allowed to return.
    maxrec=20000,

    # Declination range from which random sky positions are drawn, in degrees.
    # Avoiding the exact poles keeps the small-angle coordinate shift stable.
    min_dec_deg=-80.0,
    max_dec_deg=80.0,

    # Allowed angular separation between the two simulated images, in arcseconds.
    separation_arcsec_min=0.3,
    separation_arcsec_max=5.0,

    # True queries GOST independently for images A and B. False gives both images
    # the scanning law obtained for A. True is the more realistic option.
    query_A_and_B_separately=True,

    # Name stored in the output tables for the simulated photometric band. The
    # Amoeba generator currently uses a fixed wavelength of 620 nm internally.
    band="r_620nm",

    # Long-term standard deviation of the intrinsic quasar flux divided by its
    # mean. For example, 0.12 means approximately 12% fractional variability.
    fractional_variability=0.12,

    # DRW damping timescale, in days. This is the characteristic time over
    # which the quasar remembers its previous brightness. Smaller values give
    # faster variations; larger values give smoother, longer variations.
    drw_tau_days=300.0,

    # Range from which the true delay of every positive pair is randomly drawn.
    delay_min_days=5.0,
    delay_max_days=700.0,

    # Allowed flux ratio A/B for positive pairs. Sampling uniformly in log space
    # gives comparable representation to ratios below and above one.
    flux_ratio_min=0.2,
    flux_ratio_max=5.0,

    # Range of the overall macro-magnification assigned to image A. For positive
    # pairs, the magnification of B is derived from A and the selected flux ratio.
    macro_mag_overall_min=0.8,
    macro_mag_overall_max=1.4,

    # Range controlling the amplitude of the independent smooth microlensing
    # variation applied to each image. Larger values create stronger variations.
    microlensing_strength_min=0.02,
    microlensing_strength_max=0.18,

    # Range of smoothing timescales for microlensing, in days. Larger values
    # produce slower and smoother microlensing trends.
    microlensing_smooth_days_min=80.0,
    microlensing_smooth_days_max=350.0,

    # Hard lower and upper limits applied to the multiplicative microlensing factor.
    microlensing_clip_min=0.55,
    microlensing_clip_max=1.85,

    # Range of relative Gaussian measurement-error levels. For example, 0.04
    # produces a standard deviation equal to about 4% of the typical flux level.
    sigma_flux_frac_min=0.005,
    sigma_flux_frac_max=0.04,

    # Range of fractions of GOST observations randomly removed from each image.
    # 0.20 means that up to 20% of the available visits may be removed.
    dropout_min=0.00,
    dropout_max=0.20,

    # Types of non-lensed pairs used as negative examples. Each negative pair
    # randomly selects one of these constructions.
    negative_types=[
        "independent",
        "shifted_independent",
        "similar_confuser",
        "microlensing_confuser",
    ],
)


# ============================================================
# 6. SMALL TEST AND LARGE PRODUCTION RUNS
# ============================================================

# Small test: three positive and three negative pairs. The fixed seed makes
# random choices reproducible when the software and cached GOST data are unchanged.
CONFIG_SMALL = dict(
    **BASE_CONFIG,
    run_name="small",
    random_seed=123,
    n_positive=3,
    n_negative=3,
    output_dir=SMALL_DIR,
)

# Large run: 500 positive and 500 negative pairs. Change these two counts if a
# different class balance or dataset size is required.
CONFIG_BIG = dict(
    **BASE_CONFIG,
    run_name="big",
    random_seed=456,
    n_positive=500,
    n_negative=500,
    output_dir=BIG_DIR,
)

print("Small output:", SMALL_DIR.resolve())
print("Big output:", BIG_DIR.resolve())


## 2. Random close-pair sky positions

Positive and negative pairs both use this function, so they share the same position and separation distributions.

In [ ]:
def draw_random_sky_position(rng, min_dec_deg=-80.0, max_dec_deg=80.0):
    ra = rng.uniform(0.0, 360.0)
    s1 = np.sin(np.deg2rad(min_dec_deg))
    s2 = np.sin(np.deg2rad(max_dec_deg))
    dec = np.rad2deg(np.arcsin(rng.uniform(s1, s2)))
    return float(ra), float(dec)


def offset_radec_small_angle(ra_deg, dec_deg, separation_arcsec, position_angle_deg):
    sep_deg = separation_arcsec / 3600.0
    pa = np.deg2rad(position_angle_deg)
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 1e-6)
    ddec = sep_deg * np.cos(pa)
    dra = sep_deg * np.sin(pa) / cos_dec
    return float((ra_deg + dra) % 360.0), float(np.clip(dec_deg + ddec, -89.999, 89.999))


def draw_random_close_pair_position(config, rng):
    ra_A, dec_A = draw_random_sky_position(rng, config["min_dec_deg"], config["max_dec_deg"])
    sep_arcsec = rng.uniform(config["separation_arcsec_min"], config["separation_arcsec_max"])
    pa_deg = rng.uniform(0.0, 360.0)
    ra_B, dec_B = offset_radec_small_angle(ra_A, dec_A, sep_arcsec, pa_deg)
    return dict(
        ra_A=ra_A, dec_A=dec_A, ra_B=ra_B, dec_B=dec_B,
        separation_arcsec=float(sep_arcsec), position_angle_deg=float(pa_deg),
    )

rng_demo = np.random.default_rng(1)
for _ in range(3):
    print(draw_random_close_pair_position(CONFIG_SMALL, rng_demo))


## 3. GOST ObjVisSAP query + cache

This uses the corrected ObjVisSAP parameter names: `s_ra`, `s_dec`, `t_min`, `t_max`, `MAXREC`.

The notebook extracts the visit time from `t_start` / `t_stop` when these are present.

In [ ]:
def utc_to_mjd(date_utc):
    return float(Time(pd.to_datetime(date_utc).to_pydatetime(), scale="utc").mjd)

def mjd_to_datetime(x):
    return pd.Series(Time(np.asarray(x, dtype=float), format="mjd", scale="utc").to_datetime())

def jd_to_datetime(x):
    return pd.Series(Time(np.asarray(x, dtype=float), format="jd", scale="utc").to_datetime())

def jyear_to_datetime(x):
    return pd.Series(Time(np.asarray(x, dtype=float), format="jyear", scale="utc").to_datetime())

def clean_object_columns(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].map(lambda v: v.decode("utf-8", errors="replace") if isinstance(v, (bytes, bytearray)) else v)
    return out


def extract_gost_times(gost_df):
    cols = list(gost_df.columns)
    norm = {c: str(c).lower().replace(" ", "").replace("_", "").replace("-", "").replace("[", "").replace("]", "") for c in cols}

    start_cols = [c for c in cols if "tstart" in norm[c] or norm[c] in ["start", "starttime", "startmjd"]]
    stop_cols = [c for c in cols if "tstop" in norm[c] or norm[c] in ["stop", "stoptime", "stopmjd"]]
    if start_cols and stop_cols:
        t1 = pd.to_numeric(gost_df[start_cols[0]], errors="coerce")
        t2 = pd.to_numeric(gost_df[stop_cols[0]], errors="coerce")
        vals = (0.5 * (t1 + t2)).dropna().to_numpy()
        if len(vals) == 0:
            raise RuntimeError("GOST t_start/t_stop empty")
        med = np.nanmedian(vals)
        if med > 2_000_000:
            times = jd_to_datetime(vals)
        elif med > 40_000:
            times = mjd_to_datetime(vals)
        elif 1900 < med < 2200:
            times = jyear_to_datetime(vals)
        else:
            raise RuntimeError(f"Unknown GOST time scale; median={med}")
        return times.dropna().drop_duplicates().sort_values().reset_index(drop=True)

    for c in cols:
        name = norm[c]
        if ("date" in name or "utc" in name or "eventdate" in name or "observationtimeatgaia" in name) and not pd.api.types.is_numeric_dtype(gost_df[c]):
            times = pd.to_datetime(gost_df[c].astype(str).str.replace("(TCB)", "", regex=False).str.replace("(UTC)", "", regex=False).str.strip(), errors="coerce")
            times = pd.Series(times).dropna().drop_duplicates().sort_values().reset_index(drop=True)
            if len(times) > 0:
                return times

    for c in cols:
        if not pd.api.types.is_numeric_dtype(gost_df[c]):
            continue
        name = norm[c]
        if not ("mjd" in name or "jd" in name or "time" in name or "epoch" in name):
            continue
        vals = pd.to_numeric(gost_df[c], errors="coerce").dropna().to_numpy()
        if len(vals) == 0:
            continue
        med = np.nanmedian(vals)
        if med > 2_000_000:
            times = jd_to_datetime(vals)
        elif med > 40_000:
            times = mjd_to_datetime(vals)
        elif 1900 < med < 2200:
            times = jyear_to_datetime(vals)
        else:
            continue
        return times.dropna().drop_duplicates().sort_values().reset_index(drop=True)

    raise RuntimeError(f"Could not identify GOST time column. Columns: {cols}")


def gost_cache_path(ra_deg, dec_deg, start_utc, end_utc, maxrec):
    key = f"ra={ra_deg:.7f}|dec={dec_deg:.7f}|start={start_utc}|end={end_utc}|maxrec={maxrec}"
    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]
    return GOST_CACHE_DIR / f"gost_{digest}.csv"


def query_gost_objvissap_raw(ra_deg, dec_deg, start_utc, end_utc, maxrec=20000, timeout=180):
    url = "https://gaia.esac.esa.int/gost/ObjVisSAP/gaiaobjvisap"
    params = dict(
        s_ra=float(ra_deg),
        s_dec=float(dec_deg),
        t_min=utc_to_mjd(start_utc),
        t_max=utc_to_mjd(end_utc),
        MAXREC=int(maxrec),
    )
    headers = {"User-Agent": "python-gost-random-close-pair-dataset", "Accept": "application/x-votable+xml;charset=utf-8;serialization=TABLE"}
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    raw = r.content
    if r.status_code != 200:
        raise RuntimeError(f"GOST HTTP {r.status_code}: {r.url}\n{raw[:1000].decode('utf-8', errors='replace')}")
    preview = raw[:1000].decode("utf-8", errors="replace").lower()
    if "<html" in preview or "<!doctype html" in preview:
        raise RuntimeError(f"GOST returned HTML instead of VOTable: {r.url}")
    table = parse_single_table(BytesIO(raw)).to_table()
    return clean_object_columns(table.to_pandas()), r.url


def get_gost_times_cached(ra_deg, dec_deg, config):
    path = gost_cache_path(ra_deg, dec_deg, config["gost_start_utc"], config["gost_end_utc"], config["maxrec"])
    if path.exists():
        return pd.Series(pd.to_datetime(pd.read_csv(path)["gost_time"])).dropna().drop_duplicates().sort_values().reset_index(drop=True)

    last_error = None
    for attempt in range(MAX_GOST_RETRIES_PER_OBJECT):
        try:
            gost_df, used_url = query_gost_objvissap_raw(ra_deg, dec_deg, config["gost_start_utc"], config["gost_end_utc"], config["maxrec"])
            times = extract_gost_times(gost_df)
            start_dt = pd.to_datetime(config["gost_start_utc"])
            end_dt = pd.to_datetime(config["gost_end_utc"])
            times = times[(times >= start_dt) & (times <= end_dt)].reset_index(drop=True)
            if len(times) == 0:
                raise RuntimeError("GOST returned zero times in local window")
            pd.DataFrame({"gost_time": times}).to_csv(path, index=False)
            time.sleep(GOST_SLEEP_BETWEEN_CALLS_SEC)
            return times
        except Exception as e:
            last_error = e
            time.sleep(1.0 + attempt)
    raise RuntimeError(f"GOST failed for RA={ra_deg}, Dec={dec_deg}: {last_error}")


def times_to_simulation_days(times, start_utc):
    t0 = pd.to_datetime(start_utc)
    return (pd.Series(pd.to_datetime(times)) - t0).dt.total_seconds().to_numpy() / 86400.0


## 4. Intrinsic AGN generator: DRW by default, Amoeba optional

The default model is a **Damped Random Walk (DRW)**, also called an
Ornstein–Uhlenbeck process. For consecutive times, the exact discrete update is

$$
X_i = \phi_i X_{i-1} + \sqrt{1-\phi_i^2}\,\epsilon_i,
\qquad
\phi_i = \exp\left(-\frac{t_i-t_{i-1}}{\tau}\right),
$$

where $\epsilon_i\sim\mathcal{N}(0,1)$ and $\tau$ is `drw_tau_days`.
The simulated relative flux is

$$
F_i = 1 + f_{\mathrm{var}}X_i,
$$

where $f_{\mathrm{var}}$ is `fractional_variability`.

- A smaller `drw_tau_days` produces faster, less-correlated variations.
- A larger `drw_tau_days` produces smoother variations with longer memory.
- The same seed always reproduces the same intrinsic light curve.
- The recurrence works directly with regular or irregular time intervals.

Set `USE_AMOEBA_IF_AVAILABLE=True` only when Amoeba should be tried first. If
Amoeba is unavailable and `ALLOW_DRW_FALLBACK=True`, the notebook uses DRW.


In [ ]:
def drw_flux(
    t_days,
    seed,
    frac_var=0.12,
    tau_days=300.0,
    mean_flux=1.0,
    minimum_flux=0.05,
):
    """Generate one intrinsic quasar light curve with a DRW model.

    Parameters
    ----------
    t_days : array-like
        Strictly increasing observation times, expressed in days. The exact
        recurrence supports both regularly and irregularly spaced times.
    seed : int
        Random seed. Reusing it reproduces exactly the same light curve.
    frac_var : float, default=0.12
        Long-term standard deviation divided by the mean flux. A value of 0.12
        represents approximately 12 percent fractional variability.
    tau_days : float, default=300.0
        Damping timescale in days. It controls how long the process remembers
        its previous value. Larger values create longer, smoother variations.
    mean_flux : float, default=1.0
        Mean level around which the simulated flux varies.
    minimum_flux : float, default=0.05
        Positive numerical floor applied to the final flux.

    Returns
    -------
    numpy.ndarray
        Simulated flux at every value of ``t_days``.

    Notes
    -----
    The unit-variance stationary DRW recurrence is

        X_i = phi_i X_(i-1) + sqrt(1 - phi_i**2) epsilon_i,
        phi_i = exp(-(t_i - t_(i-1)) / tau_days),

    with epsilon_i drawn independently from a standard normal distribution.
    The returned flux is mean_flux * (1 + frac_var * X_i).
    """

    # Convert the input into one predictable numerical representation.
    t_days = np.asarray(t_days, dtype=float)

    # Validate every public argument before beginning the simulation. Clear
    # error messages make incorrect notebook settings easier to diagnose.
    if t_days.ndim != 1:
        raise ValueError("t_days must be a one-dimensional array.")
    if len(t_days) == 0:
        raise ValueError("t_days must contain at least one value.")
    if not np.all(np.isfinite(t_days)):
        raise ValueError("Every value in t_days must be finite.")
    if len(t_days) > 1 and np.any(np.diff(t_days) <= 0.0):
        raise ValueError("t_days must be strictly increasing without duplicates.")
    if not np.isfinite(tau_days) or tau_days <= 0.0:
        raise ValueError("tau_days must be a positive finite number.")
    if not np.isfinite(frac_var) or frac_var < 0.0:
        raise ValueError("frac_var must be a non-negative finite number.")
    if not np.isfinite(mean_flux) or mean_flux <= 0.0:
        raise ValueError("mean_flux must be a positive finite number.")
    if not np.isfinite(minimum_flux) or minimum_flux <= 0.0:
        raise ValueError("minimum_flux must be a positive finite number.")

    # default_rng provides an isolated and reproducible random-number stream.
    rng = np.random.default_rng(int(seed))
    process = np.empty(len(t_days), dtype=float)

    # Starting from N(0, 1) places the DRW directly in its stationary
    # distribution and avoids a special low-variability beginning.
    process[0] = rng.normal(loc=0.0, scale=1.0)

    for index in range(1, len(t_days)):
        delta_t = t_days[index] - t_days[index - 1]

        # phi is the fraction of the previous value retained after delta_t.
        phi = np.exp(-delta_t / tau_days)

        # This scale keeps the stationary variance of process equal to one.
        innovation_sigma = np.sqrt(max(0.0, 1.0 - phi**2))
        process[index] = (
            phi * process[index - 1]
            + innovation_sigma * rng.normal(loc=0.0, scale=1.0)
        )

    # Translate the unit-variance process into the requested relative flux.
    flux = mean_flux * (1.0 + frac_var * process)

    # At the default 12 percent variability this floor is rarely reached, but
    # it guarantees positive flux for extreme user-selected settings.
    return np.clip(flux, minimum_flux, None)


class IntrinsicAGNGenerator:
    """Select Amoeba when requested; otherwise generate intrinsic DRW curves."""

    def __init__(self, config):
        """Prepare the generator described by the notebook configuration."""

        self.config = config
        self.mode = "drw"

        # The following attributes are used only by the optional Amoeba model.
        self.disk = None
        self.smbh_mass_exp = 8.5
        self.redshift_source = 0.5
        self.band_wavelength_nm = 620.0

        # The recommended default follows this branch and needs no optional
        # dependency beyond NumPy.
        if not USE_AMOEBA_IF_AVAILABLE:
            print(
                "Intrinsic generator: DRW "
                f"(tau = {self.config['drw_tau_days']:.1f} days)"
            )
            return

        # Amoeba is imported only when requested, so a normal DRW run does not
        # require the optional amoeba-agn package to be installed.
        try:
            from amoeba.Util.util import create_maps
            from amoeba.Classes.accretion_disk import AccretionDisk

            disk_kwargs = create_maps(
                smbh_mass_exp=self.smbh_mass_exp,
                redshift_source=self.redshift_source,
                number_grav_radii=800,
                inclination_angle=30,
                resolution=350,
                eddington_ratio=0.1,
                spin=0.0,
                temp_beta=1.0,
                generic_beta=True,
            )
            self.disk = AccretionDisk(**disk_kwargs)
            self.mode = "amoeba"
            print("Intrinsic generator: Amoeba")

        except Exception as error:
            if not ALLOW_DRW_FALLBACK:
                raise

            self.mode = "drw"
            print("Amoeba is unavailable; using the DRW fallback.")
            print("Amoeba error:", repr(error))

    def generate(self, t_days, seed):
        """Return one reproducible intrinsic light curve on ``t_days``."""

        t_days = np.asarray(t_days, dtype=float)
        n_points = len(t_days)

        if self.mode == "amoeba":
            try:
                from amoeba.Util.util import (
                    convolve_signal_with_transfer_function,
                    generate_signal_from_psd,
                )

                frequencies = np.linspace(
                    1.0 / (2.0 * max(n_points, 2)),
                    1.0 / 2.0,
                    n_points,
                )
                power_spectral_density = frequencies ** (-2.0)
                _, driving_signal = generate_signal_from_psd(
                    n_points,
                    power_spectral_density,
                    frequencies,
                    int(seed),
                )

                driving_signal = np.asarray(driving_signal, dtype=float)
                driving_std = np.std(driving_signal)
                if not np.isfinite(driving_std) or driving_std == 0.0:
                    raise RuntimeError("Amoeba returned an invalid driving signal.")
                driving_signal = (
                    driving_signal - np.mean(driving_signal)
                ) / driving_std

                transfer_function = (
                    self.disk.construct_accretion_disk_transfer_function(
                        self.band_wavelength_nm
                    )
                )
                _, light_curve = convolve_signal_with_transfer_function(
                    smbh_mass_exp=self.smbh_mass_exp,
                    driving_signal=driving_signal,
                    transfer_function=transfer_function,
                    initial_time_axis=np.arange(n_points, dtype=float),
                    redshift_source=self.redshift_source,
                    desired_cadence_in_days=self.config["cadence_days"],
                )

                light_curve = np.asarray(light_curve, dtype=float)[:n_points]
                light_curve_std = np.std(light_curve)
                if not np.isfinite(light_curve_std) or light_curve_std == 0.0:
                    raise RuntimeError("Amoeba returned an invalid light curve.")
                light_curve = (
                    light_curve - np.mean(light_curve)
                ) / light_curve_std

                flux = 1.0 + self.config["fractional_variability"] * light_curve
                return np.clip(flux, 0.05, None)

            except Exception as error:
                if not ALLOW_DRW_FALLBACK:
                    raise

                print("Amoeba generation failed; using DRW for this curve.")
                print("Amoeba error:", repr(error))

        # This is both the default execution path and the Amoeba fallback.
        return drw_flux(
            t_days=t_days,
            seed=seed,
            frac_var=self.config["fractional_variability"],
            tau_days=self.config["drw_tau_days"],
            mean_flux=1.0,
            minimum_flux=0.05,
        )


## 5. Microlensing and sampling utilities

In [ ]:
def generate_smooth_microlensing(t_days, seed, strength=0.1, smooth_days=200.0, clip_min=0.55, clip_max=1.85):
    rng = np.random.default_rng(seed)
    t_days = np.asarray(t_days, dtype=float)
    n = len(t_days)
    dt = np.median(np.diff(t_days)) if n > 1 else 1.0
    sigma_samples = max(1.0, smooth_days / dt)
    trend = gaussian_filter1d(rng.normal(size=n), sigma=sigma_samples, mode="reflect")
    for _ in range(rng.integers(1, 4)):
        center = rng.uniform(t_days.min(), t_days.max())
        width = rng.uniform(0.15, 0.45) * (t_days.max() - t_days.min())
        trend += rng.normal() * np.exp(-0.5 * ((t_days - center) / width) ** 2)
    trend = trend - np.mean(trend)
    if not np.isfinite(np.std(trend)) or np.std(trend) == 0:
        trend = rng.normal(size=n)
    trend = trend / np.std(trend)
    mu = 1.0 + strength * trend
    mu = gaussian_filter1d(mu, sigma=max(1.0, 0.15 * sigma_samples), mode="reflect")
    mu = np.clip(mu, clip_min, clip_max)
    return mu / np.mean(mu)


def interp_curve(tq, t, y):
    return np.interp(tq, t, y, left=np.nan, right=np.nan)


def robust_noise_sigma(values, sigma_frac):
    scale = np.nanmedian(np.abs(values))
    if not np.isfinite(scale) or scale <= 0:
        scale = np.nanstd(values)
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    return float(sigma_frac * scale)


def sample_one_image(pair_id, image_id, t_cont, flux_cont, gost_times, config, rng, sigma_frac, dropout_frac):
    sim_days = times_to_simulation_days(gost_times, config["gost_start_utc"])
    mask = (sim_days >= t_cont.min()) & (sim_days <= t_cont.max())
    sim_days = sim_days[mask]
    gost_times_used = pd.Series(pd.to_datetime(gost_times))[mask].reset_index(drop=True)
    if len(sim_days) == 0:
        return pd.DataFrame()
    true = interp_curve(sim_days, t_cont, flux_cont)
    ok = np.isfinite(true)
    sim_days, true = sim_days[ok], true[ok]
    gost_times_used = gost_times_used[ok].reset_index(drop=True)
    keep = rng.random(len(true)) >= dropout_frac
    sim_days, true = sim_days[keep], true[keep]
    gost_times_used = gost_times_used[keep].reset_index(drop=True)
    if len(true) == 0:
        return pd.DataFrame()
    sigma = robust_noise_sigma(true, sigma_frac)
    obs = true + rng.normal(0.0, sigma, size=len(true))
    return pd.DataFrame(dict(pair_id=pair_id, image_id=image_id, gost_time=gost_times_used, time_days=sim_days, flux_true=true, flux_obs=obs, flux_err=np.full(len(true), sigma)))


def make_continuous_dataframe(pair_id, label, t, flux_A, flux_B, mu_A, mu_B, src_A, src_B):
    return pd.concat([
        pd.DataFrame(dict(pair_id=pair_id, label_lensed=label, image_id="A", time_days=t, intrinsic_component=src_A, microlensing_mu=mu_A, flux_continuous=flux_A)),
        pd.DataFrame(dict(pair_id=pair_id, label_lensed=label, image_id="B", time_days=t, intrinsic_component=src_B, microlensing_mu=mu_B, flux_continuous=flux_B)),
    ], ignore_index=True)


## 6. Pair generation

For positives, both images share the same intrinsic source with a true delay.

For negatives, A and B are different intrinsic sources, but they still use close sky positions and GOST scanning laws just like positives.

In [ ]:
def get_pair_gost_times(sky, config):
    times_A = get_gost_times_cached(sky["ra_A"], sky["dec_A"], config)
    if config.get("query_A_and_B_separately", True):
        times_B = get_gost_times_cached(sky["ra_B"], sky["dec_B"], config)
    else:
        times_B = times_A.copy()
    return times_A, times_B


def common_microlensing_and_noise(config, rng, t_cont, negative_type=None):
    if negative_type == "microlensing_confuser":
        strength_A = rng.uniform(0.12, 0.30)
        strength_B = rng.uniform(0.12, 0.30)
    else:
        strength_A = rng.uniform(config["microlensing_strength_min"], config["microlensing_strength_max"])
        strength_B = rng.uniform(config["microlensing_strength_min"], config["microlensing_strength_max"])
    smooth_A = rng.uniform(config["microlensing_smooth_days_min"], config["microlensing_smooth_days_max"])
    smooth_B = rng.uniform(config["microlensing_smooth_days_min"], config["microlensing_smooth_days_max"])
    mu_A = generate_smooth_microlensing(t_cont, int(rng.integers(1, 2_000_000_000)), strength_A, smooth_A, config["microlensing_clip_min"], config["microlensing_clip_max"])
    mu_B = generate_smooth_microlensing(t_cont, int(rng.integers(1, 2_000_000_000)), strength_B, smooth_B, config["microlensing_clip_min"], config["microlensing_clip_max"])
    sigma_A = rng.uniform(config["sigma_flux_frac_min"], config["sigma_flux_frac_max"])
    sigma_B = rng.uniform(config["sigma_flux_frac_min"], config["sigma_flux_frac_max"])
    dropout_A = rng.uniform(config["dropout_min"], config["dropout_max"])
    dropout_B = rng.uniform(config["dropout_min"], config["dropout_max"])
    return mu_A, mu_B, strength_A, strength_B, smooth_A, smooth_B, sigma_A, sigma_B, dropout_A, dropout_B


def generate_positive_pair(pair_index, config, rng, gen):
    pair_id = f"pos_{pair_index:06d}"
    sky = draw_random_close_pair_position(config, rng)
    times_A, times_B = get_pair_gost_times(sky, config)
    t_cont = np.arange(0.0, config["n_days"] + config["cadence_days"], config["cadence_days"])
    delay = rng.uniform(config["delay_min_days"], config["delay_max_days"])
    pad = config["max_delay_days"] + 30.0
    t_src = np.arange(-pad, config["n_days"] + config["cadence_days"], config["cadence_days"])
    source_seed = int(rng.integers(1, 2_000_000_000))
    source = gen.generate(t_src, source_seed)
    src_A = interp_curve(t_cont, t_src, source)
    src_B = interp_curve(t_cont - delay, t_src, source)
    ratio = 10 ** rng.uniform(np.log10(config["flux_ratio_min"]), np.log10(config["flux_ratio_max"]))
    macro_A = rng.uniform(config["macro_mag_overall_min"], config["macro_mag_overall_max"])
    macro_B = macro_A / ratio
    mu_A, mu_B, sA, sB, smA, smB, sigA, sigB, dropA, dropB = common_microlensing_and_noise(config, rng, t_cont)
    flux_A = macro_A * src_A * mu_A
    flux_B = macro_B * src_B * mu_B
    obs_A = sample_one_image(pair_id, "A", t_cont, flux_A, times_A, config, rng, sigA, dropA)
    obs_B = sample_one_image(pair_id, "B", t_cont, flux_B, times_B, config, rng, sigB, dropB)
    if len(obs_A) < 5 or len(obs_B) < 5:
        raise RuntimeError("Too few sampled points")
    obs = pd.concat([obs_A, obs_B], ignore_index=True)
    obs["label_lensed"] = 1
    obs["band"] = config["band"]
    cont = make_continuous_dataframe(pair_id, 1, t_cont, flux_A, flux_B, mu_A, mu_B, src_A, src_B)
    meta = dict(pair_id=pair_id, label_lensed=1, negative_type="none", source_seed_A=source_seed, source_seed_B=source_seed, true_delay_days=float(delay), confuser_shift_days=np.nan, flux_ratio_A_over_B=float(ratio), macro_mag_A=float(macro_A), macro_mag_B=float(macro_B), microlensing_strength_A=float(sA), microlensing_strength_B=float(sB), microlensing_smooth_days_A=float(smA), microlensing_smooth_days_B=float(smB), sigma_flux_frac_A=float(sigA), sigma_flux_frac_B=float(sigB), dropout_A=float(dropA), dropout_B=float(dropB), n_obs_A=int(len(obs_A)), n_obs_B=int(len(obs_B)), **sky)
    return meta, obs, cont


def generate_negative_pair(pair_index, config, rng, gen):
    pair_id = f"neg_{pair_index:06d}"
    sky = draw_random_close_pair_position(config, rng)
    times_A, times_B = get_pair_gost_times(sky, config)
    t_cont = np.arange(0.0, config["n_days"] + config["cadence_days"], config["cadence_days"])
    pad = config["max_delay_days"] + 30.0
    t_src = np.arange(-pad, config["n_days"] + config["cadence_days"], config["cadence_days"])
    negative_type = str(rng.choice(config["negative_types"]))
    seed_A = int(rng.integers(1, 2_000_000_000))
    seed_B = int(rng.integers(1, 2_000_000_000))
    source_A = gen.generate(t_src, seed_A)
    source_B = gen.generate(t_src, seed_B)
    fake_shift = 0.0
    if negative_type in ["shifted_independent", "similar_confuser"]:
        fake_shift = rng.uniform(config["delay_min_days"], config["delay_max_days"])
    src_A = interp_curve(t_cont, t_src, source_A)
    src_B = interp_curve(t_cont - fake_shift, t_src, source_B)
    macro_A = rng.uniform(config["macro_mag_overall_min"], config["macro_mag_overall_max"])
    macro_B = rng.uniform(config["macro_mag_overall_min"], config["macro_mag_overall_max"])
    ratio = macro_A / macro_B
    mu_A, mu_B, sA, sB, smA, smB, sigA, sigB, dropA, dropB = common_microlensing_and_noise(config, rng, t_cont, negative_type=negative_type)
    flux_A = macro_A * src_A * mu_A
    flux_B = macro_B * src_B * mu_B
    obs_A = sample_one_image(pair_id, "A", t_cont, flux_A, times_A, config, rng, sigA, dropA)
    obs_B = sample_one_image(pair_id, "B", t_cont, flux_B, times_B, config, rng, sigB, dropB)
    if len(obs_A) < 5 or len(obs_B) < 5:
        raise RuntimeError("Too few sampled points")
    obs = pd.concat([obs_A, obs_B], ignore_index=True)
    obs["label_lensed"] = 0
    obs["band"] = config["band"]
    cont = make_continuous_dataframe(pair_id, 0, t_cont, flux_A, flux_B, mu_A, mu_B, src_A, src_B)
    meta = dict(pair_id=pair_id, label_lensed=0, negative_type=negative_type, source_seed_A=seed_A, source_seed_B=seed_B, true_delay_days=np.nan, confuser_shift_days=float(fake_shift), flux_ratio_A_over_B=float(ratio), macro_mag_A=float(macro_A), macro_mag_B=float(macro_B), microlensing_strength_A=float(sA), microlensing_strength_B=float(sB), microlensing_smooth_days_A=float(smA), microlensing_smooth_days_B=float(smB), sigma_flux_frac_A=float(sigA), sigma_flux_frac_B=float(sigB), dropout_A=float(dropA), dropout_B=float(dropB), n_obs_A=int(len(obs_A)), n_obs_B=int(len(obs_B)), **sky)
    return meta, obs, cont


## 7. Dataset builder

In [ ]:
def build_dataset(config):
    outdir = Path(config["output_dir"])
    outdir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(config["random_seed"])
    gen = IntrinsicAGNGenerator(config)
    metas, obs_parts, cont_parts = [], [], []

    def build_class(label, n_target):
        generated, attempts = 0, 0
        while generated < n_target:
            attempts += 1
            if attempts > n_target * MAX_SKY_TRIES_PER_PAIR:
                raise RuntimeError(f"Too many failures for label={label}")
            try:
                if label == 1:
                    meta, obs, cont = generate_positive_pair(generated, config, rng, gen)
                else:
                    meta, obs, cont = generate_negative_pair(generated, config, rng, gen)
                metas.append(meta)
                obs_parts.append(obs)
                cont_parts.append(cont)
                generated += 1
                print(f"[{config['run_name']}] label={label} {generated}/{n_target}: {meta['pair_id']} sep={meta['separation_arcsec']:.2f} arcsec")
            except Exception as e:
                print(f"[{config['run_name']}] retry label={label}; reason: {e}")

    build_class(1, config["n_positive"])
    build_class(0, config["n_negative"])

    metadata = pd.DataFrame(metas)
    observations = pd.concat(obs_parts, ignore_index=True)
    continuous = pd.concat(cont_parts, ignore_index=True)

    # Store the intrinsic-model choice with the pair-level metadata so a
    # saved dataset remains scientifically reproducible without reopening
    # this notebook. drw_tau_days is unused only when Amoeba succeeds.
    metadata["intrinsic_generator"] = gen.mode
    metadata["drw_tau_days"] = float(config["drw_tau_days"])

    metadata.to_csv(outdir / "metadata.csv", index=False)
    observations.to_csv(outdir / "observations.csv", index=False)
    continuous.to_csv(outdir / "continuous_curves.csv", index=False)
    with open(outdir / "config.json", "w", encoding="utf-8") as f:
        json.dump({k: str(v) if isinstance(v, Path) else v for k, v in config.items()}, f, indent=2)

    print("Saved to", outdir.resolve())
    return metadata, observations, continuous


## 8. Plots: every small pair + microlensing

In [ ]:
def plot_pair(pair_id, metadata, observations, continuous, save_dir=None):
    meta = metadata.loc[metadata["pair_id"] == pair_id].iloc[0]
    obs = observations[observations["pair_id"] == pair_id]
    cont = continuous[continuous["pair_id"] == pair_id]
    label = int(meta["label_lensed"])
    label_txt = "positive lensed" if label == 1 else f"negative: {meta['negative_type']}"

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
    for image_id, marker in [("A", "o"), ("B", "s")]:
        c = cont[cont["image_id"] == image_id]
        o = obs[obs["image_id"] == image_id]
        axes[0].plot(c["time_days"], c["flux_continuous"], lw=1.8, label=f"image {image_id} continuous")
        axes[0].errorbar(o["time_days"], o["flux_obs"], yerr=o["flux_err"], fmt=marker, ms=4, capsize=2, alpha=0.85, label=f"image {image_id} GOST sampled")
        axes[1].plot(c["time_days"], c["microlensing_mu"], lw=2.0, label=f"mu {image_id}")
        axes[2].plot(c["time_days"], c["intrinsic_component"], lw=1.5, label=f"intrinsic component {image_id}")

    axes[0].set_title(f"{pair_id} — {label_txt}\nsep={meta['separation_arcsec']:.2f} arcsec | A=({meta['ra_A']:.4f},{meta['dec_A']:.4f}) B=({meta['ra_B']:.4f},{meta['dec_B']:.4f})")
    axes[0].set_ylabel("Flux")
    axes[0].legend()
    axes[1].axhline(1.0, ls="--", lw=1)
    axes[1].set_ylabel("Microlensing mu")
    axes[1].legend()
    axes[2].set_ylabel("Intrinsic component")
    axes[2].set_xlabel("Time from GOST start [days]")
    axes[2].legend()
    if label == 1:
        txt = f"true delay={meta['true_delay_days']:.1f} d | flux ratio A/B={meta['flux_ratio_A_over_B']:.2f}"
    else:
        txt = f"fake/confuser shift={meta['confuser_shift_days']:.1f} d"
    axes[0].text(0.01, 0.03, txt, transform=axes[0].transAxes)
    plt.tight_layout()
    if save_dir is not None:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(save_dir) / f"{pair_id}.png", dpi=160, bbox_inches="tight")
    plt.show()


def plot_all_small_pairs(metadata, observations, continuous, output_dir):
    plot_dir = Path(output_dir) / "pair_plots"
    for pair_id in metadata["pair_id"]:
        plot_pair(pair_id, metadata, observations, continuous, save_dir=plot_dir)
    print("Saved pair plots in", plot_dir.resolve())


def plot_dataset_qc(metadata, observations, title="Dataset QC"):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()
    metadata["label_lensed"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
    axes[0].set_title("Labels")
    axes[1].hist(metadata["separation_arcsec"], bins=20)
    axes[1].set_title("Separation [arcsec]")
    axes[2].hist(metadata["dec_A"], bins=20)
    axes[2].set_title("Dec distribution")
    pos = metadata[metadata["label_lensed"] == 1]
    if len(pos):
        axes[3].hist(pos["true_delay_days"].dropna(), bins=20)
    axes[3].set_title("True delays, positives")
    axes[4].hist(metadata["n_obs_A"], bins=20, alpha=0.7, label="A")
    axes[4].hist(metadata["n_obs_B"], bins=20, alpha=0.7, label="B")
    axes[4].set_title("N observations")
    axes[4].legend()
    dts = []
    for _, g in observations.sort_values("time_days").groupby(["pair_id", "image_id"]):
        dts.extend(np.diff(g["time_days"].to_numpy()))
    if len(dts):
        axes[5].hist(dts, bins=40)
    axes[5].set_title("GOST cadence")
    axes[5].set_xlabel("days")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## 9. One quasar, one chosen delay, several GOST samplings

This optional use case generates **one intrinsic quasar light curve**, creates image B by applying the **same user-selected delay**, and then samples that unchanged pair with as many different random GOST scanning laws as requested.

Each sampling realisation draws a new random close pair of sky positions and queries GOST for A and B. The intrinsic source and true delay remain identical in every realisation, which makes this cell useful for studying the effect of Gaia sampling alone. Run Sections 1–8 before executing it.

In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

# Delay applied to image B, in days. The convention is:
#
#     flux_B(t) = flux_A(t - USER_DELAY_DAYS)
#
# A positive value means that image B is observed later than image A.
USER_DELAY_DAYS = 120.0

# Number of different GOST sampling realisations to generate. Each
# realisation uses a new random sky position but the same quasar and delay.
USER_N_GOST_SAMPLINGS = 5

# Seeds controlling reproducibility. USER_SOURCE_SEED fixes the intrinsic
# quasar curve; USER_RANDOM_SEED fixes sky positions and measurement noise.
USER_SOURCE_SEED = 2027
USER_RANDOM_SEED = 2026

# Relative Gaussian measurement-error level applied after GOST sampling.
# 0.02 corresponds to a standard deviation of about 2% of the typical flux.
USER_NOISE_FRACTION = 0.02

# Fraction of GOST visits randomly removed after sampling. Use 0.0 to retain
# every available visit and isolate the effect of the scanning law.
USER_DROPOUT_FRACTION = 0.0

# Reject a sampling realisation if A or B has fewer observations than this.
USER_MIN_POINTS_PER_IMAGE = 5

# Number of generated realisations displayed in the diagnostic figure.
# The CSV files still contain every requested realisation. Use 0 for no plot.
USER_PLOT_FIRST_N = 3

# Directory receiving the continuous curve, sampling metadata, and sampled data.
USER_OUTPUT_DIR = OUTPUT_ROOT / "fixed_delay_gost_realisations"


def gost_sampling_signature(times_A, times_B):
    """Return a short identifier for one pair of GOST time sequences."""

    values_A = np.asarray(pd.to_datetime(times_A), dtype="datetime64[ns]").astype("int64")
    values_B = np.asarray(pd.to_datetime(times_B), dtype="datetime64[ns]").astype("int64")
    digest = hashlib.sha1()
    digest.update(values_A.tobytes())
    digest.update(b"|A-B|")
    digest.update(values_B.tobytes())
    return digest.hexdigest()[:16]


def generate_fixed_delay_gost_realisations(
    delay_days,
    n_gost_samplings,
    source_seed,
    random_seed,
    noise_fraction,
    dropout_fraction,
    min_points_per_image,
    output_dir,
    plot_first_n=3,
):
    """Generate one delayed quasar pair under several GOST samplings."""

    delay_days = float(delay_days)
    n_gost_samplings = int(n_gost_samplings)
    min_points_per_image = int(min_points_per_image)
    output_dir = Path(output_dir)

    if not np.isfinite(delay_days):
        raise ValueError("delay_days must be a finite number.")
    if n_gost_samplings < 1:
        raise ValueError("n_gost_samplings must be at least 1.")
    if min_points_per_image < 1:
        raise ValueError("min_points_per_image must be at least 1.")
    if not 0.0 <= float(dropout_fraction) < 1.0:
        raise ValueError("dropout_fraction must be between 0 and 1.")
    if float(noise_fraction) < 0.0:
        raise ValueError("noise_fraction cannot be negative.")

    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(int(random_seed))

    # Cover the complete GOST date interval so no returned visit is discarded
    # simply because the original 2,000-day BASE_CONFIG curve was too short.
    config = dict(BASE_CONFIG)
    gost_duration_days = (
        pd.to_datetime(config["gost_end_utc"])
        - pd.to_datetime(config["gost_start_utc"])
    ).total_seconds() / 86400.0
    config["n_days"] = float(gost_duration_days)

    # Generate extra source history on both sides. This works for positive and
    # negative delays and does not depend on BASE_CONFIG['max_delay_days'].
    cadence = float(config["cadence_days"])
    padding = abs(delay_days) + 30.0
    t_continuous = np.arange(0.0, config["n_days"] + cadence, cadence)
    t_source = np.arange(-padding, config["n_days"] + padding + cadence, cadence)

    generator = IntrinsicAGNGenerator(config)
    source_flux = generator.generate(t_source, int(source_seed))

    # Both images come from this same intrinsic source. Only B is shifted. No
    # microlensing or macro-magnification is added in this controlled use case.
    flux_A = interp_curve(t_continuous, t_source, source_flux)
    flux_B = interp_curve(t_continuous - delay_days, t_source, source_flux)

    continuous = pd.DataFrame(
        {
            "time_days": t_continuous,
            "flux_A": flux_A,
            "flux_B": flux_B,
            "true_delay_days": delay_days,
            "source_seed": int(source_seed),
            "intrinsic_generator": generator.mode,
            "drw_tau_days": float(config["drw_tau_days"]),
        }
    )

    metadata_rows = []
    observation_parts = []
    used_signatures = set()

    for sampling_index in range(n_gost_samplings):
        sampling_generated = False
        last_error = None

        for attempt in range(MAX_SKY_TRIES_PER_PAIR):
            try:
                sky = draw_random_close_pair_position(config, rng)
                times_A, times_B = get_pair_gost_times(sky, config)
                signature = gost_sampling_signature(times_A, times_B)

                # If two random positions return exactly the same pair of visit
                # sequences, draw another position so all outputs are distinct.
                if signature in used_signatures:
                    continue

                sampling_id = f"gost_{sampling_index:04d}"
                obs_A = sample_one_image(
                    sampling_id,
                    "A",
                    t_continuous,
                    flux_A,
                    times_A,
                    config,
                    rng,
                    float(noise_fraction),
                    float(dropout_fraction),
                )
                obs_B = sample_one_image(
                    sampling_id,
                    "B",
                    t_continuous,
                    flux_B,
                    times_B,
                    config,
                    rng,
                    float(noise_fraction),
                    float(dropout_fraction),
                )

                if len(obs_A) < min_points_per_image or len(obs_B) < min_points_per_image:
                    raise RuntimeError("Too few usable GOST observations.")

                observations = pd.concat([obs_A, obs_B], ignore_index=True)
                observations["sampling_index"] = sampling_index
                observations["gost_signature"] = signature
                observations["true_delay_days"] = delay_days
                observations["source_seed"] = int(source_seed)
                observations["band"] = config["band"]
                observation_parts.append(observations)

                metadata_rows.append(
                    {
                        "sampling_id": sampling_id,
                        "sampling_index": sampling_index,
                        "gost_signature": signature,
                        "true_delay_days": delay_days,
                        "source_seed": int(source_seed),
                        "intrinsic_generator": generator.mode,
                        "drw_tau_days": float(config["drw_tau_days"]),
                        "n_obs_A": len(obs_A),
                        "n_obs_B": len(obs_B),
                        **sky,
                    }
                )

                used_signatures.add(signature)
                sampling_generated = True
                print(
                    f"Sampling {sampling_index + 1}/{n_gost_samplings}: "
                    f"{len(obs_A)} visits for A, {len(obs_B)} visits for B"
                )
                break

            except Exception as error:
                last_error = error

        if not sampling_generated:
            raise RuntimeError(
                f"Could not generate GOST sampling {sampling_index}. "
                f"Last error: {last_error}"
            )

    metadata = pd.DataFrame(metadata_rows)
    sampled_observations = pd.concat(observation_parts, ignore_index=True)

    continuous.to_csv(output_dir / "continuous_delayed_pair.csv", index=False)
    metadata.to_csv(output_dir / "gost_sampling_metadata.csv", index=False)
    sampled_observations.to_csv(
        output_dir / "gost_sampled_observations.csv",
        index=False,
    )

    saved_config = {
        "delay_days": delay_days,
        "n_gost_samplings": n_gost_samplings,
        "source_seed": int(source_seed),
        "random_seed": int(random_seed),
        "noise_fraction": float(noise_fraction),
        "dropout_fraction": float(dropout_fraction),
        "min_points_per_image": min_points_per_image,
        "gost_start_utc": config["gost_start_utc"],
        "gost_end_utc": config["gost_end_utc"],
        "generator_mode": generator.mode,
        "drw_tau_days": float(config["drw_tau_days"]),
    }
    with (output_dir / "configuration.json").open("w", encoding="utf-8") as stream:
        json.dump(saved_config, stream, indent=2)

    n_to_plot = min(max(int(plot_first_n), 0), n_gost_samplings)
    if n_to_plot > 0:
        figure, axes = plt.subplots(
            n_to_plot,
            1,
            figsize=(13, 4 * n_to_plot),
            sharex=True,
            squeeze=False,
        )

        for plot_index in range(n_to_plot):
            axis = axes[plot_index, 0]
            sampling_id = metadata.iloc[plot_index]["sampling_id"]
            current = sampled_observations[
                sampled_observations["pair_id"] == sampling_id
            ]

            axis.plot(t_continuous, flux_A, lw=1.2, alpha=0.55, label="A continuous")
            axis.plot(t_continuous, flux_B, lw=1.2, alpha=0.55, label="B continuous")

            for image_id, marker in [("A", "o"), ("B", "s")]:
                points = current[current["image_id"] == image_id]
                axis.errorbar(
                    points["time_days"],
                    points["flux_obs"],
                    yerr=points["flux_err"],
                    fmt=marker,
                    ms=3,
                    capsize=2,
                    alpha=0.8,
                    label=f"{image_id} GOST sampled",
                )

            axis.set_title(f"{sampling_id} — fixed true delay = {delay_days:.2f} days")
            axis.set_ylabel("Flux")
            axis.grid(alpha=0.25)
            axis.legend(ncol=2)

        axes[-1, 0].set_xlabel("Days since the beginning of the GOST interval")
        plt.tight_layout()
        plt.show()

    print("Saved fixed-delay experiment to:", output_dir.resolve())
    return {
        "continuous": continuous,
        "metadata": metadata,
        "observations": sampled_observations,
        "configuration": saved_config,
    }


fixed_delay_result = generate_fixed_delay_gost_realisations(
    delay_days=USER_DELAY_DAYS,
    n_gost_samplings=USER_N_GOST_SAMPLINGS,
    source_seed=USER_SOURCE_SEED,
    random_seed=USER_RANDOM_SEED,
    noise_fraction=USER_NOISE_FRACTION,
    dropout_fraction=USER_DROPOUT_FRACTION,
    min_points_per_image=USER_MIN_POINTS_PER_IMAGE,
    output_dir=USER_OUTPUT_DIR,
    plot_first_n=USER_PLOT_FIRST_N,
)

display(fixed_delay_result["metadata"])


## 10. Run small implementation

In [ ]:
if RUN_SMALL:
    small_metadata, small_observations, small_continuous = build_dataset(CONFIG_SMALL)
    display(small_metadata)
    plot_dataset_qc(small_metadata, small_observations, title="Small dataset QC")
    plot_all_small_pairs(small_metadata, small_observations, small_continuous, CONFIG_SMALL["output_dir"])
else:
    print("RUN_SMALL=False")


## 11. Run big implementation

Set `RUN_BIG=True` in the config cell when the small run looks correct. The big run will query many random close sky pairs, so it can take time.

In [ ]:
if RUN_BIG:
    big_metadata, big_observations, big_continuous = build_dataset(CONFIG_BIG)
    display(big_metadata.head())
    plot_dataset_qc(big_metadata, big_observations, title="Big dataset QC")
else:
    print("RUN_BIG=False. Set RUN_BIG=True in the first config cell when ready.")
